In [0]:
# Read Files
from pyspark.sql.functions import *

path = "/Volumes/rg_azure_databricks/week_4/file"
products_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(f"{path}/products.csv")
sales_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(f"{path}/sales.csv")

display(products_df)
display(sales_df)

product_id,product_name,category,brand,cost_price,selling_price
P001,Laptop,Electronics,Dell,45000,55000
P002,Smartphone,Electronics,Samsung,18000,25000
P003,Headphones,Electronics,Sony,1200,2000
P004,TShirt,Clothing,Levis,400,800
P005,Jeans,Clothing,Levis,900,1800
P006,Sneakers,Footwear,Nike,2000,3500
P007,Rice,Grocery,IndiaGate,45,60
P008,Cooking Oil,Grocery,Fortune,110,150
P009,Chocolate,Food,Cadbury,20,40
P010,Coffee Powder,Beverages,Nescafe,150,250


sale_id,store_id,product_id,sale_date,quantity,discount_percent
S001,ST01,P001,2025-01-05,5,10
S002,ST01,P002,2025-01-05,12,5
S003,ST02,P003,2025-01-06,25,0
S004,ST03,P004,2025-01-06,40,15
S005,ST01,P005,2025-01-07,18,10
S006,ST02,P006,2025-01-07,14,5
S007,ST03,P007,2025-01-08,120,2
S008,ST01,P008,2025-01-08,90,3
S009,ST02,P009,2025-01-09,150,0
S010,ST03,P010,2025-01-09,60,5


In [0]:
# Clean Data
products_df = products_df.dropDuplicates()
sales_df = sales_df.dropDuplicates()
sales_df = sales_df.na.fill({
    "quantity":0,
    "discount_percent":0
})

In [0]:
# Join Product + Sales
retail_df = sales_df.join(
    products_df,
    "product_id",
    "inner"
)

display(retail_df)

product_id,sale_id,store_id,sale_date,quantity,discount_percent,product_name,category,brand,cost_price,selling_price
P011,S026,ST01,2025-01-17,5,8,Television,Electronics,LG,25000,32000
P002,S047,ST01,2025-01-28,18,5,Smartphone,Electronics,Samsung,18000,25000
P003,S048,ST02,2025-01-28,40,0,Headphones,Electronics,Sony,1200,2000
P011,S011,ST01,2025-01-10,4,8,Television,Electronics,LG,25000,32000
P015,S030,ST02,2025-01-19,60,5,Water Bottle,Accessories,Milton,80,150
P013,S028,ST03,2025-01-18,80,0,Shampoo,PersonalCare,Dove,120,220
P008,S038,ST01,2025-01-23,110,3,Cooking Oil,Grocery,Fortune,110,150
P008,S008,ST01,2025-01-08,90,3,Cooking Oil,Grocery,Fortune,110,150
P006,S006,ST02,2025-01-07,14,5,Sneakers,Footwear,Nike,2000,3500
P007,S037,ST03,2025-01-23,140,2,Rice,Grocery,IndiaGate,45,60


In [0]:
# Revenue Calculation
retail_df = retail_df.withColumn(
    "revenue",
    round(
        col("quantity") *
        col("selling_price") *
        (1 - col("discount_percent")/100),
        2
    )
)
display(retail_df.select("product_id","product_name","quantity","selling_price","discount_percent","revenue"))

product_id,product_name,quantity,selling_price,discount_percent,revenue
P011,Television,5,32000,8,147200.0
P002,Smartphone,18,25000,5,427500.0
P003,Headphones,40,2000,0,80000.0
P011,Television,4,32000,8,117760.0
P015,Water Bottle,60,150,5,8550.0
P013,Shampoo,80,220,0,17600.0
P008,Cooking Oil,110,150,3,16005.0
P008,Cooking Oil,90,150,3,13095.0
P006,Sneakers,14,3500,5,46550.0
P007,Rice,140,60,2,8232.0


In [0]:
# Profit Calculation
retail_df = retail_df.withColumn(
    "profit",
    round(
        col("revenue") -
        (col("quantity") * col("cost_price")),
        2
    )
)

display(retail_df.select("product_id","product_name","quantity","cost_price","revenue","profit"))

product_id,product_name,quantity,cost_price,revenue,profit
P011,Television,5,25000,147200.0,22200.0
P002,Smartphone,18,18000,427500.0,103500.0
P003,Headphones,40,1200,80000.0,32000.0
P011,Television,4,25000,117760.0,17760.0
P015,Water Bottle,60,80,8550.0,3750.0
P013,Shampoo,80,120,17600.0,8000.0
P008,Cooking Oil,110,110,16005.0,3905.0
P008,Cooking Oil,90,110,13095.0,3195.0
P006,Sneakers,14,2000,46550.0,18550.0
P007,Rice,140,45,8232.0,1932.0


In [0]:
# Product Metrics
product_metrics = retail_df.groupBy(
    "product_name",
    "category"
).agg(
    sum("quantity").alias("units_sold"),
    round(sum("revenue"),2).alias("revenue"),
    round(sum("profit"),2).alias("profit")
)

display(product_metrics)

product_name,category,units_sold,revenue,profit
Water Bottle,Accessories,180,25650.0,11250.0
TShirt,Clothing,205,137400.0,55400.0
Laptop,Electronics,26,1280400.0,110400.0
Television,Electronics,15,441600.0,66600.0
Smartphone,Electronics,55,1306250.0,316250.0
Coffee Powder,Beverages,195,46312.5,17062.5
Sneakers,Footwear,48,159600.0,63600.0
Soap,PersonalCare,630,18900.0,9450.0
Chocolate,Food,500,20000.0,10000.0
Cooking Oil,Grocery,300,43650.0,10650.0


In [0]:
# Category Profit Margin
category_metrics = retail_df.groupBy(
    "category"
).agg(
    round(sum("revenue"),2).alias("revenue"),
    round(sum("profit"),2).alias("profit")
)

category_metrics = category_metrics.withColumn(
    "profit_margin",
    round(
        (col("profit")/col("revenue"))*100,
        2
    )
)

display(category_metrics)

category,revenue,profit,profit_margin
Food,20000.0,10000.0,50.0
Grocery,66582.0,16032.0,24.08
Electronics,3547450.0,640450.0,18.05
Clothing,288060.0,122360.0,42.48
Footwear,159600.0,63600.0,39.85
Beverages,46312.5,17062.5,36.84
PersonalCare,71700.0,33450.0,46.65
Accessories,25650.0,11250.0,43.86


In [0]:
# Store Metrics
store_metrics = retail_df.groupBy(
    "store_id"
).agg(
    round(sum("revenue"),2).alias("revenue"),
    round(sum("profit"),2).alias("profit"),
    sum("quantity").alias("units_sold")
)

display(store_metrics)

store_id,revenue,profit,units_sold
ST01,2208560.0,492410.0,1098
ST03,1292344.5,189744.5,1051
ST02,724450.0,232050.0,870


In [0]:
# Dashboard Dataset
dashboard_df = retail_df.groupBy(
    "store_id",
    "category"
).agg(
    round(sum("revenue"),2).alias("total_revenue"),
    round(sum("profit"),2).alias("total_profit"),
    sum("quantity").alias("units_sold")
)

display(dashboard_df)

store_id,category,total_revenue,total_profit,units_sold
ST03,Beverages,46312.5,17062.5,195
ST01,PersonalCare,18900.0,9450.0,630
ST03,PersonalCare,52800.0,24000.0,240
ST03,Clothing,137400.0,55400.0,205
ST03,Electronics,1032900.0,87900.0,21
ST02,Electronics,519200.0,147200.0,142
ST02,Accessories,25650.0,11250.0,180
ST01,Electronics,1995350.0,405350.0,75
ST01,Clothing,150660.0,66960.0,93
ST02,Footwear,159600.0,63600.0,48


In [0]:
# Save as Delta Tables
retail_df.write.format("delta").mode("overwrite").saveAsTable("retail_sales")
product_metrics.write.format("delta").mode("overwrite").saveAsTable("product_metrics")
category_metrics.write.format("delta").mode("overwrite").saveAsTable("category_metrics")
store_metrics.write.format("delta").mode("overwrite").saveAsTable("store_metrics")

In [0]:
# Create a TempView
retail_df.createOrReplaceTempView("retail_sales")

In [0]:
%sql
-- Product By Total Units Sold
SELECT product_name, SUM(quantity) AS total_units_sold FROM retail_sales
GROUP BY product_name
ORDER BY total_units_sold DESC
LIMIT 3;

product_name,total_units_sold
Soap,630
Chocolate,500
Rice,390


In [0]:
%sql
-- Lowest Performing Products
SELECT product_name, ROUND(SUM(revenue),2) AS total_revenue FROM retail_sales
GROUP BY product_name
ORDER BY total_revenue ASC
LIMIT 5;

product_name,total_revenue
Soap,18900.0
Chocolate,20000.0
Rice,22932.0
Water Bottle,25650.0
Cooking Oil,43650.0
